In [30]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Tabular: [jam_tidur, aktivitas_fisik, jam_kerja]
X_tabular = np.array([
    [6.0, 1.0, 8.0],
    [3.0, 0.0, 12.0],
    [8.0, 2.0, 5.0]
])

# Label tabular: 0 = no, 1 = yes
y_tab = np.array([1, 1, 0])
y_tab_onehot = tf.keras.utils.to_categorical(y_tab, num_classes=2)

In [31]:
# Teks: catatan harian
X_text = np.array([
    "Hari ini capek banget, tapi semua kerjaan beres sih",
    "Mati rasa, tugas banyak yang belum selesai, dosen galak, pengen nyerah deh",
    "Santai banget hari ini, asik juga bisa mabar dan nongki"
], dtype=object)

# Label teks: 0 = anxiety, 1 = depression, 2 = normal, 3 = suicidal
y_txt = np.array([0, 1, 2])
y_txt_onehot = tf.keras.utils.to_categorical(y_txt, num_classes=4)

In [32]:
input_tab = layers.Input(shape=(3,), name="input_tab")
x1 = layers.Dense(16, activation='relu')(input_tab)
out_tab = layers.Dense(2, activation='softmax', name="out_tab")(x1)
model_tab = models.Model(input_tab, out_tab)

model_tab.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
model_tab.fit(X_tabular, y_tab_onehot, epochs=10, verbose=0)

In [33]:
vectorize_layer = layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=20
)
vectorize_layer.adapt(X_text)

input_txt = layers.Input(shape=(1,), dtype=tf.string, name="input_txt")
x2 = vectorize_layer(input_txt)
x2 = layers.Embedding(1000, 16)(x2)
x2 = layers.LSTM(16)(x2)
out_txt = layers.Dense(4, activation='softmax', name="out_txt")(x2)
model_txt = models.Model(input_txt, out_txt)

model_txt.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
model_txt.fit(X_text, y_txt_onehot, epochs=10, verbose=0)

In [34]:
prob_tab = model_tab.predict(X_tabular)
prob_txt = model_txt.predict(X_text)

print("Probabilitas Tabular (No Stress | Stress):\n", prob_tab)
print("\nProbabilitas Teks (anxiety | depression | normal | suicidal):\n", prob_txt)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
Probabilitas Tabular (No Stress | Stress):
 [[0.01385936 0.9861406 ]
 [0.00287877 0.99712116]
 [0.01780502 0.98219496]]

Probabilitas Teks (anxiety | depression | normal | suicidal):
 [[0.26085204 0.25445148 0.2641597  0.22053674]
 [0.26036507 0.25431186 0.2634792  0.22184385]
 [0.2605281  0.25411582 0.265131   0.22022516]]


In [35]:
w_tab = 0.4
w_txt = 0.6

stress_score_tab = prob_tab[:, 1]

adjustment = np.stack([
    stress_score_tab,
    stress_score_tab,
    1 - stress_score_tab,
    stress_score_tab
], axis=1)

adjustment = adjustment / adjustment.sum(axis=1, keepdims=True)

final_prob = (w_tab * adjustment) + (w_txt * prob_txt)
final_prob = final_prob / final_prob.sum(axis=1, keepdims=True)

final_class = np.argmax(final_prob, axis=1)
label_map = {0: "anxiety", 1: "depression", 2: "normal", 3: "suicidal"}

In [36]:
print("\n── Hasil Late Fusion ──")
for i in range(len(final_class)):
    print(f"Sampel {i+1}: {label_map[final_class[i]]:12s} | "
          f"prob={final_prob[i].round(3)} | "
          f"tab_stress={stress_score_tab[i]:.2f}")

print(f"\nPrediksi kelas akhir (index): {final_class}")


── Hasil Late Fusion ──
Sampel 1: anxiety      | prob=[0.289 0.285 0.16  0.265] | tab_stress=0.99
Sampel 2: anxiety      | prob=[0.289 0.286 0.158 0.266] | tab_stress=1.00
Sampel 3: anxiety      | prob=[0.289 0.285 0.161 0.265] | tab_stress=0.98

Prediksi kelas akhir (index): [0 0 0]
